In [ ]:
"""

Title: "Scalable Neural Safety Certification via Monotonicity"
Code for: Learing a neural barrier certificate for Freeway traffic flow dynamics example.
Author: Amirreza Alavi
Contact: seyedamirreza.alavi@colorado.edu
Affiliation: University of Colorado Boulder
Date: 04/2026

"""

In [ ]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers, constraints, initializers
import matplotlib.pyplot as plt
from numpy import linalg as LA
from scipy import stats
from functools import partial
%matplotlib inline
import numpy as np
from scipy.optimize import fsolve
from tensorflow.keras.layers import Input, Dense, Concatenate, Add
from tensorflow.keras import Model
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model
import time
from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D  
from math import inf
import heapq
from dataclasses import dataclass
import matplotlib.patches as mpatches

In [ ]:
# System Definition

t1 = time.perf_counter()

def phi(x):
    return tf.constant(15,dtype=tf.float32)*(1-tf.exp(-0.07*tf.constant(x,dtype=tf.float32)))

def sigma(x):
    return  tf.maximum(0,tf.constant(15,dtype=tf.float32)*(1- tf.pow(x/tf.constant(50,dtype=tf.float32),3)))

def f_m(q):
    
    if not isinstance(q, (tf.Tensor, tf.Variable)):
        q = tf.convert_to_tensor(q, dtype=tf.float32)
    else:
        q = tf.cast(q, tf.float32)
    if not isinstance(q, tf.Variable):
        q = tf.Variable(q, trainable=False)  

    fq  = tf.Variable(tf.zeros_like(q), trainable=True)
    tau = tf.constant(0.5, dtype=tf.float32)

    for i in range(q.shape[0]):
        fq[i].assign(
            q[i] + tau * (
                tf.minimum(phi(tf.roll(q[i], shift=1, axis=0)), sigma(tf.roll(q[i], shift=0, axis=0)))
                - tf.minimum(phi(q[i]), sigma(tf.roll(q[i], shift=-1, axis=0)))
            )
        )

        fq[i, 0].assign(
            q[i, 0] + tau * (7.5 - tf.minimum(phi(q[i, 0]), sigma(q[i, 1])))
        )
        fq[i, 1].assign(
            q[i, 1] + tau * (
                tf.minimum(phi(q[i, 0]), sigma(q[i, 1])) - tf.minimum(phi(q[i, 1]), sigma(q[i, 2]))
            )
        )
        fq[i, -1].assign(
            q[i, -1] + tau * (
                tf.minimum(phi(q[i, -2]), sigma(q[i, -1])) - phi(q[i, -1])
            )
        )
    return tf.clip_by_value(fq, 0.0, 50)


In [ ]:
# Sampled points for a coarse gridding

N=1000

xin_bar = np.ones((1,N))*5
xin_und = np.zeros((1,N))
xuns_und = np.eye(N)*40
xuns_bar = np.ones((1,N))*50
xi_bar= np.ones((1,N))*40
xi_und= np.zeros((1,N))

In [ ]:
# System Inductive evaluation for K=50000

def f_m_0km1(q, k):
    q = tf.convert_to_tensor(q, dtype=tf.float32)
    if q.shape.rank == 1:
        q = tf.expand_dims(q, 0)     
    B, N = q.shape
    traj = tf.TensorArray(tf.float32, size=k, element_shape=(B, N))
    for k in tf.range(k):
        traj = traj.write(k, q)      
        q = f_m(q)                  
    return traj.stack() 


def f_m_k(q, k):
    x = tf.convert_to_tensor(q, dtype=tf.float32)
    for _ in tf.range(k):
        x = f_m(x)
    return x

k=50000

f_m_0km1_xin_bar = tf.squeeze(f_m_0km1(xin_bar,k))
f_m_0km1_xi_und = tf.squeeze(f_m_0km1(xi_und,k), axis=1)
f_m_k_xi_bar = f_m_k(xi_bar,k)

In [ ]:
# Neural Barrier Definition

class ScalarMonotoneNN(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        if activation is None:
            activation = tf.keras.layers.LeakyReLU(alpha=0.1)

        self.hidden_layers = []
        for h in hidden_dims:
            self.hidden_layers.append(
                layers.Dense(
                    h,
                    activation=activation,
                    use_bias=True,
                    kernel_constraint=constraints.NonNeg()
                )
            )

        self.out_layer = layers.Dense(
            1,
            activation=output_activation,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

    def call(self, x):
        z = x
        for layer in self.hidden_layers:
            z = layer(z)
        return self.out_layer(z)


class MaxAffineBranch(layers.Layer):

    def __init__(self, m_i, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        self.affine = layers.Dense(
            m_i,
            activation=None,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        z = self.affine(x)                            
        g = tf.reduce_max(z, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                       
        return y


class InputMaxBranch(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        g = tf.reduce_max(x, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                       
        return y


class ThreeBranchMinMonotoneNetwork(Model):

    def __init__(self,
                 input_dim,
                 m1,
                 m2,
                 hidden_dims1=(200, 200),
                 hidden_dims2=(200, 200),
                 hidden_dims3=(200, 200),
                 activation=None,
                 output_activation='linear'):
        super().__init__()

        self.input_dim = input_dim

        self.branch1 = InputMaxBranch(
            hidden_dims=hidden_dims1,
            activation=activation,
            output_activation=output_activation
        )

        self.branch2 = MaxAffineBranch(
            m_i=m1,
            hidden_dims=hidden_dims2,
            activation=activation,
            output_activation=output_activation
        )

        self.branch3 = MaxAffineBranch(
            m_i=m2,
            hidden_dims=hidden_dims3,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        y1 = self.branch1(x)   
        y2 = self.branch2(x)   
        y3 = self.branch3(x)   

        Y = tf.concat([y1, y2, y3], axis=1)          
        y = tf.reduce_min(Y, axis=1, keepdims=True)  
        return y


modelB = ThreeBranchMinMonotoneNetwork(
    input_dim=N,
    m1=int(0.1*N),   
    m2=int(0.2*N),   
    hidden_dims1=(200, 200),     
    hidden_dims2=(200, 200),
    hidden_dims3=(200, 200),
    activation=tf.keras.layers.LeakyReLU(alpha=0.1),
    output_activation='linear'
)

In [ ]:
# Training Monotone Neural Inductive Barrier

epochs = 100000
opt1 = tf.keras.optimizers.Adam(1e-2)

gamma = -20
eta = 20


for i in range(epochs):
    with tf.GradientTape() as tape0, tf.GradientTape() as tape1:

        loss_B = tf.reduce_sum(tf.nn.relu(tf.subtract(1.0,1.0)))

        loss_B += tf.square(tf.nn.relu(tf.reduce_max(modelB(f_m_0km1_xin_bar))- gamma))
        loss_B += tf.square(tf.nn.relu(eta-tf.reduce_min(modelB(xuns_und))))
        loss_B += tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_m_0km1_xi_und)) - 9.9)))  * (tf.nn.relu(modelB(f_m_k_xi_bar) -  9.9))

        third_loss_B = tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_m_0km1_xi_und)) - 9.9)))  * (tf.nn.relu(modelB(f_m_k_xi_bar) -  9.9))

        third_loss_B_val = third_loss_B.numpy().item()

                                
        grad_B = tape0.gradient(loss_B, modelB.trainable_variables)
                
        if loss_B == 0   and i>1:
            print('Safety Verified')
            break
            
        if loss_B!=0:
            opt1.apply_gradients(zip(grad_B, modelB.trainable_variables),)
            
        if i%5 == 1:
            print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_m_0km1_xin_bar)).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.4f}, "
            f"Third condition = {third_loss_B_val:.4f}, "
            f"LossB = {loss_B.numpy().item():.4f}"
            )


    
t2 = time.perf_counter()
tot = t2 - t1
print(f'Total runtime: {tot}s')
print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_m_0km1_xin_bar)).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.4f}, "
            f"Third condition = {third_loss_B_val:.4f}, "
            f"LossB = {loss_B.numpy().item():.4f}"
            )